# Lab 16 — Access Controls for Cortex AI

Control who can use which AI models and whether cross-region inference is allowed.

| Item | Detail |
|---|---|
| **Parameters** | `CORTEX_MODELS_ALLOWLIST`, `CORTEX_ENABLED_CROSS_REGION` |
| **Exam Domain** | 3.0 Gen AI Governance (26%) |
| **You'll learn** | Model allowlists, RBAC for AI, cross-region policy, Network Rules |


## Step 1 — Environment Setup

In [ ]:
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — CORTEX_MODELS_ALLOWLIST

This account-level parameter restricts which Cortex models are available.
Default is empty (all models allowed).


In [ ]:
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
-- Example: restrict to only these models (requires ACCOUNTADMIN)
-- ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-3-5-haiku,mistral-large2,e5-base-v2';

-- View current setting
SELECT SYSTEM$GET_TAG_ON_CURRENT_TABLE('CORTEX_MODELS_ALLOWLIST');

## Step 3 — Cross-Region Inference

`CORTEX_ENABLED_CROSS_REGION` controls whether AI inference can run in other Snowflake regions.

| Setting | Behavior |
|---|---|
| `DISABLED` (default) | Models only run in your account's region |
| `ENABLED` | Models can run in any region if local capacity is unavailable |

**Data governance implication:** Cross-region may affect data residency compliance.


In [ ]:
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

## Step 4 — RBAC for Cortex AI

Cortex AI functions require specific privileges:

| Privilege | What It Grants |
|---|---|
| `USAGE` on DATABASE | Access to Cortex functions in that DB |
| Role-based model access | Via `CORTEX_MODELS_ALLOWLIST` per role (future) |
| Cortex Search | `USAGE` on the search service |
| Semantic views | `SELECT` on the semantic view |


In [ ]:
-- Example: Create a role with limited Cortex access
CREATE ROLE IF NOT EXISTS CORTEX_READER;
GRANT USAGE ON DATABASE GENAI_LEARNING TO ROLE CORTEX_READER;
GRANT USAGE ON SCHEMA GENAI_LEARNING.PUBLIC TO ROLE CORTEX_READER;
GRANT SELECT ON ALL TABLES IN SCHEMA GENAI_LEARNING.PUBLIC TO ROLE CORTEX_READER;
GRANT SELECT ON ALL VIEWS IN SCHEMA GENAI_LEARNING.PUBLIC TO ROLE CORTEX_READER;

## Step 5 — Network Rules & External Access

For Cortex features that access external resources (e.g., REST APIs):
- **Network Rules** control which external endpoints are reachable
- **External Access Integrations** package network rules + secrets for UDFs/procedures


## Key Takeaways

- `CORTEX_MODELS_ALLOWLIST` restricts available models (account-level, ACCOUNTADMIN only).
- `CORTEX_ENABLED_CROSS_REGION` controls data residency for AI inference.
- Standard Snowflake RBAC (roles, grants) governs access to Cortex objects.
- Network Rules + External Access Integrations control external connectivity.
- These are heavily tested in Exam Domain 3.0 (26% of exam).
